# Session 1: two almost identical starts drift apart
**A. Real-world problem.** A grandfather clock swings the same way every time: if you know
how it starts, you know where it will be in an hour. Hang a second pendulum from the end of
the first and the motion looks random. Is it really random? Run the same experiment twice,
with starting angles that differ by one hundred-millionth of a radian, and see.

**Plan.** (1) simulate a single pendulum, (2) simulate a double pendulum, (3) start two
copies almost identically, (4) measure how far apart they drift.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
G = 9.81      # gravity, m/s^2
LEN = 1.0     # every rod is 1 m long, every ball is 1 kg

## Step 1: a single pendulum
The angle `theta` is measured from hanging straight down. Gravity pulls it back with

    theta'' = -(g / L) * sin(theta)       (rad/s^2)

For small swings sin(theta) is about theta and the swing time (period) is T = 2 pi sqrt(L / g).
For L = 1 m that is 2.006 s.

We step forward in tiny time steps `h` with the Runge-Kutta 4 rule (RK4): it looks at the
slope four times per step and takes a clever average. Session 2 explains why it is accurate.

In [ ]:
def single_rate(state):
    """state = (theta, omega). Returns (theta', omega') for one pendulum."""
    theta, omega = state
    return np.array([omega, -(G / LEN) * np.sin(theta)])


def rk4_step(rate, state, h):
    """One RK4 step of size h (s) for any rate function."""
    k1 = rate(state)
    k2 = rate(state + 0.5 * h * k1)
    k3 = rate(state + 0.5 * h * k2)
    k4 = rate(state + h * k3)
    return state + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate(rate, state0, duration, h=0.005):
    """Return the times (s) and an array of states, one row per time."""
    n = int(round(duration / h))
    states = np.empty((n + 1, len(state0)))
    states[0] = state0
    for i in range(n):
        states[i + 1] = rk4_step(rate, states[i], h)
    return np.arange(n + 1) * h, states


def single_period(theta0, h=0.001):
    """Swing time (s) measured from the simulation: the time for the pendulum to come back to
    its release point, found where omega changes from positive to negative (first time it does)."""
    t, s = simulate(single_rate, [theta0, 0.0], 6.0, h)
    om = s[:, 1]
    # released with omega < 0 starting; omega goes + to - only back at the release side
    idx = np.where((om[:-1] > 0) & (om[1:] <= 0))[0]
    i = idx[0]
    return float(t[i] + h * om[i] / (om[i] - om[i + 1]))

## Step 2: a double pendulum
Now a second rod hangs from the first ball. There are two angles, `theta1` (upper rod) and
`theta2` (lower rod), and two angular speeds. For equal balls and equal rods the two angular
accelerations are (Session 2 derives them):

    d = theta1 - theta2
    theta1'' = [ -3 g sin(theta1) - g sin(theta1 - 2 theta2) - 2 sin(d) (omega2^2 + omega1^2 cos d) ] / (3 - cos 2d)
    theta2'' = [ 2 sin(d) (2 omega1^2 + 2 g cos(theta1) + omega2^2 cos d) ] / (3 - cos 2d)

In [ ]:
def double_rate(state):
    """state = (theta1, theta2, omega1, omega2). Equal masses, equal 1 m rods."""
    t1, t2, w1, w2 = state
    d = t1 - t2
    den = 3.0 - np.cos(2 * d)
    a1 = (-3 * G * np.sin(t1) - G * np.sin(t1 - 2 * t2) - 2 * np.sin(d) * (w2 ** 2 + w1 ** 2 * np.cos(d))) / den
    a2 = (2 * np.sin(d) * (2 * w1 ** 2 + 2 * G * np.cos(t1) + w2 ** 2 * np.cos(d))) / den
    return np.array([w1, w2, a1, a2])


def tip_positions(states):
    """Positions (x, y) in metres of the elbow ball and the end ball; y is up, pivot at (0, 0)."""
    t1, t2 = states[:, 0], states[:, 1]
    x1, y1 = LEN * np.sin(t1), -LEN * np.cos(t1)
    x2, y2 = x1 + LEN * np.sin(t2), y1 - LEN * np.cos(t2)
    return x1, y1, x2, y2

## Step 3: two almost identical starts
Release the double pendulum from rest with both rods at 120 degrees. Make a twin whose
upper angle is larger by only 1e-8 rad (about the width of a hair compared with the size of
the Earth). Then measure the distance between the two states as time goes on.

In [ ]:
START = np.radians(120.0)


def twin_gap(rate, state0, delta, duration, h=0.005, index=0):
    """Absolute difference (rad) between the angle `index` of two runs whose first angle starts delta apart."""
    other = np.array(state0, dtype=float)
    other[0] += delta
    t, a = simulate(rate, state0, duration, h)
    _, b = simulate(rate, other, duration, h)
    return t, np.abs(a[:, index] - b[:, index]), a


def first_time_above(t, gap, limit):
    """First time (s) at which gap exceeds limit, or None if it never does."""
    idx = np.where(gap > limit)[0]
    return float(t[idx[0]]) if len(idx) else None

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print(f"single pendulum, small swing (2 degrees): period {single_period(np.radians(2)):.3f} s "
          f"(formula 2 pi sqrt(L/g) = {2 * np.pi * np.sqrt(LEN / G):.3f} s)")
    delta = 1e-8
    t, gap1, _ = twin_gap(single_rate, [START, 0.0], delta, 20.0)
    t, gap2, run = twin_gap(double_rate, [START, START, 0.0, 0.0], delta, 20.0, index=1)
    print(f"start difference {delta:g} rad")
    for tt in (5, 10, 15, 20):
        i = int(tt / 0.005)
        print(f"  t = {tt:2d} s   single pendulum gap {gap1[i]:.2e} rad   double pendulum gap {gap2[i]:.2e} rad")
    print("double pendulum gap first exceeds 0.1 rad at t =", first_time_above(t, gap2, 0.1), "s")

    # Figure 1: the end-ball path of the two twin double pendulums
    other = np.array([START, START, 0.0, 0.0])
    other[0] += delta
    _, run_b = simulate(double_rate, other, 20.0, 0.005)
    fig, axs = plt.subplots(1, 2, figsize=(9, 4.5), sharex=True, sharey=True)
    for ax, k, name in ((axs[0], 8, "first 8 s"), (axs[1], 20, "first 20 s")):
        n = int(k / 0.005)
        _, _, xa, ya = tip_positions(run[:n])
        _, _, xb, yb = tip_positions(run_b[:n])
        ax.plot(xa, ya, lw=0.8, label="start A")
        ax.plot(xb, yb, lw=0.8, label="start B (1e-8 rad away)")
        ax.set_title(f"End-ball path, {name}")
        ax.set_aspect("equal")
        ax.grid(alpha=0.3)
        ax.set_xlabel("x (m)")
    axs[0].set_ylabel("y (m)")
    axs[0].legend(fontsize=8, loc="lower left")
    fig.tight_layout()
    fig.savefig(OUT / "twin_paths.png", dpi=120)

    # Figure 2: how the gap grows
    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    ax2.semilogy(t, np.maximum(gap1, 1e-16), label="single pendulum")
    ax2.semilogy(t, np.maximum(gap2, 1e-16), label="double pendulum")
    ax2.axhline(0.1, color="red", ls="--", label="0.1 rad: visibly different")
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("angle gap between the twins (rad)")
    ax2.set_title("A tiny difference stays tiny, or explodes")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "gap_growth.png", dpi=120)

## Where this stops
The double pendulum is not random: the rules are exact and the computer is exact enough.
But a one-hundred-millionth error becomes a whole swing after a few seconds. Session 2 builds
the equations and the checks that tell us the simulation is right; Session 3 measures the
speed of the drift (the Lyapunov exponent).

## Coding tasks
1. **Easy.** Change `START` in `lesson.py` to 30 degrees and predict, before running, whether the double pendulum gap will pass 0.1 rad within 20 s; then check.
2. **Stretch.** Loop over release angles 10, 20, ..., 170 degrees and print for each the first time the gap passes 0.1 rad (use `first_time_above`). Which release angles never get there?
3. **Explain.** Explain in your own words why a straight line on the log graph means exponential growth, and what the slope of that line tells you.

## Assessment
- **A. Conceptual.** A friend says: the double pendulum moves randomly, so no equation can describe it. Explain what is right and what is wrong in that sentence.
- **B. Numerical.** Find the swing time of a pendulum 0.25 m long with the formula T = 2 pi sqrt(L / g). Is it more or less than the 1 m pendulum, and by what factor?
- **C. Graph and simulation interpretation.** On the gap graph, the double pendulum's gap grows from 3.2e-4 rad at 10 s to 0.62 rad at 15 s. By what factor does it grow in those 5 s, and what is the growth rate per second?
- **D. Design and optimisation challenge.** Design: you may measure the starting angle with a protractor of precision 1e-3 rad. Using a growth rate of 1.5 per second, how long can you forecast to within 0.1 rad? What precision would 10 s need? State your assumption.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    period = L.single_period(np.radians(2.0))
    hand = 2 * np.pi * np.sqrt(L.LEN / L.G)
    checks.append((f"worked example: small-swing period of a 1 m pendulum is 2 pi sqrt(L/g) = 2.006 s (got {period:.3f} s)",
                   abs(hand - 2.0061) < 1e-3 and abs(period - hand) < 0.005))

    # energy of the single pendulum (per kg, L = 1 m): E = omega^2 / 2 - g cos(theta)
    t, s = L.simulate(L.single_rate, [np.radians(120.0), 0.0], 20.0, 0.005)
    e = 0.5 * s[:, 1] ** 2 - L.G * np.cos(s[:, 0])
    checks.append(("single pendulum keeps its energy over 20 s (change below 1e-7 J/kg)", float(np.ptp(e)) < 1e-7))

    checks.append(("a double pendulum hanging at rest has zero acceleration",
                   bool(np.allclose(L.double_rate(np.array([0.0, 0.0, 0.0, 0.0])), 0.0))))

    # hand calculation of positions: both rods hanging, and both horizontal
    x1, y1, x2, y2 = L.tip_positions(np.array([[0.0, 0.0, 0.0, 0.0], [np.pi / 2, np.pi / 2, 0.0, 0.0]]))
    checks.append(("hand calculation: hanging balls at (0, -1) and (0, -2) m; horizontal balls at (1, 0) and (2, 0) m",
                   bool(np.allclose([x1[0], y1[0], x2[0], y2[0]], [0, -1, 0, -2]) and np.allclose([x1[1], y1[1], x2[1], y2[1]], [1, 0, 2, 0]))))
    _, s = L.simulate(L.double_rate, [2.0, 1.0, 0.0, 0.0], 5.0, 0.005)
    xa, ya, xb, yb = L.tip_positions(s)
    checks.append(("rods keep their 1 m length", bool(np.allclose(np.hypot(xa, ya), 1.0) and np.allclose(np.hypot(xb - xa, yb - ya), 1.0))))

    # predictable versus chaotic
    d = 1e-8
    t1, g1, _ = L.twin_gap(L.single_rate, [L.START, 0.0], d, 20.0)
    checks.append(("single pendulum: a 1e-8 rad difference stays below 1e-5 rad after 20 s", float(g1.max()) < 1e-5))
    t2, g2, _ = L.twin_gap(L.double_rate, [L.START, L.START, 0.0, 0.0], d, 20.0, index=1)
    first = L.first_time_above(t2, g2, 0.1)
    checks.append((f"double pendulum: the 1e-8 rad difference passes 0.1 rad between 10 and 16 s (got {first} s)",
                   first is not None and 10.0 < first < 16.0))
    checks.append(("double pendulum: the gap has grown by more than a factor of 1e7 by 20 s", float(g2[-1]) > 1e7 * d))

    # edge cases
    checks.append(("edge: first_time_above returns None when the limit is never reached", L.first_time_above(t1, g1, 1.0) is None))
    t0, g0, _ = L.twin_gap(L.double_rate, [0.0, 0.0, 0.0, 0.0], 0.0, 2.0)
    checks.append(("edge: zero difference gives zero gap", float(g0.max()) == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
